In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)
import re


In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- evaluate_coassembly_edges ---
FIX_EVALUATE_COASSEMBLY_EDGES_ELUSIVE_CLUSTERS_PD = pd.DataFrame(
    {"coassembly": ["c1", "c1", "c2"]},
    index=pd.Index(["s1", "s2", "s3"], name="sample"),
)
FIX_EVALUATE_COASSEMBLY_EDGES_ELUSIVE_CLUSTERS_PL = pl.DataFrame({
    "sample1": ["s1", "s2", "s3"],
    "sample2": ["s1", "s2", "s3"],
    "coassembly": ["c1", "c1", "c2"],
})
FIX_EVALUATE_COASSEMBLY_EDGES_ELUSIVE_EDGES_PD = pd.DataFrame({
    "sample1": ["s1.1", "s2.1"],
    "sample2": ["s2.1", "s3.1"],
    "target_ids": ["t1,t2", "t3"],
})
FIX_EVALUATE_COASSEMBLY_EDGES_ELUSIVE_EDGES_PL = pl.from_pandas(FIX_EVALUATE_COASSEMBLY_EDGES_ELUSIVE_EDGES_PD)
FIX_EVALUATE_COASSEMBLY_EDGES_ELUSIVE_CLUSTERS = FIX_EVALUATE_COASSEMBLY_EDGES_ELUSIVE_CLUSTERS_PD
FIX_EVALUATE_COASSEMBLY_EDGES_ELUSIVE_EDGES = FIX_EVALUATE_COASSEMBLY_EDGES_ELUSIVE_EDGES_PD

# --- evaluate_combined ---
FIX_EVALUATE_COMBINED_HAYSTACK_OTU_TABLE_PD = pd.DataFrame(
    {"target": ["t1", None, "t3"], "taxonomy": ["tax_old_1", "tax_old_2", "tax_old_3"]},
    index=pd.MultiIndex.from_tuples(
        [("c1", "g1", "seq1"), ("c2", "g2", "seq2"), ("c3", "g3", "seq3")],
        names=["coassembly", "gene", "sequence"],
    ),
)
FIX_EVALUATE_COMBINED_HAYSTACK_OTU_TABLE_PL = pl.DataFrame({
    "coassembly": ["c1", "c2", "c3"],
    "gene": ["g1", "g2", "g3"],
    "sequence": ["seq1", "seq2", "seq3"],
    "target": ["t1", None, "t3"],
    "taxonomy": ["tax_old_1", "tax_old_2", "tax_old_3"],
})
FIX_EVALUATE_COMBINED_RECOVERED_COASSEMBLIES = ["c1", "c2", "c3"]
FIX_EVALUATE_COMBINED_RECOVERED_OTU_TABLE_PD = pd.DataFrame({
    "coassembly": ["c1", "c2", "c3"],
    "gene": ["g1", "g2", "g3"],
    "sequence": ["seq1", "seq2", "seq3"],
    "genome": ["gm1", None, "gm3"],
    "taxonomy": ["tax1", None, "tax3"],
})
FIX_EVALUATE_COMBINED_RECOVERED_OTU_TABLE_PL = pl.from_pandas(FIX_EVALUATE_COMBINED_RECOVERED_OTU_TABLE_PD)
FIX_EVALUATE_COMBINED_HAYSTACK_OTU_TABLE = FIX_EVALUATE_COMBINED_HAYSTACK_OTU_TABLE_PD
FIX_EVALUATE_COMBINED_RECOVERED_OTU_TABLE = FIX_EVALUATE_COMBINED_RECOVERED_OTU_TABLE_PD
FIX_EVALUATE_COMBINED_A = 0

# --- evaluate_nontarget ---
FIX_EVALUATE_NONTARGET_ELUSIVE_CLUSTERS_PD = pd.DataFrame(
    {"coassembly": ["c1", "c2", "c3"]},
    index=pd.Index(["s1", "s2", "s3"], name="sample"),
)
FIX_EVALUATE_NONTARGET_ELUSIVE_CLUSTERS_PL = pl.DataFrame({"sample": ["s1", "s2", "s3"], "coassembly": ["c1", "c2", "c3"]})
FIX_EVALUATE_NONTARGET_ELUSIVE_OTU_TABLE_PD = pd.DataFrame(
    {"target": ["t1", "t2"], "taxonomy": ["tax1", "tax2"]},
    index=pd.MultiIndex.from_tuples(
        [("c1", "g1", "seq1"), ("c2", "g2", "seq2")],
        names=["coassembly", "gene", "sequence"],
    ),
)
FIX_EVALUATE_NONTARGET_ELUSIVE_OTU_TABLE_PL = pl.DataFrame(
    {
        "sample": [None, None],
        "gene": ["g1", "g2"],
        "sequence": ["seq1", "seq2"],
        "taxonomy": ["tax1", "tax2"],
        "found_in": [None, None],
        "coassembly": ["c1", "c2"],
        "target": ["t1", "t2"],
    },
    schema={
        "sample": pl.String,
        "gene": pl.String,
        "sequence": pl.String,
        "taxonomy": pl.String,
        "found_in": pl.String,
        "coassembly": pl.String,
        "target": pl.String,
    },
)
FIX_EVALUATE_NONTARGET_BINNED_OTU_TABLE_PD = pd.DataFrame({
    "sample": ["s1.1", "s2.1", "s3.1"],
    "gene": ["g1", "g2", "g3"],
    "sequence": ["seq1", "seq2", "seq3"],
    "taxonomy": ["tax1", "tax2", "tax3"],
    "found_in": ["s1", "s2", "s3"],
})
FIX_EVALUATE_NONTARGET_BINNED_OTU_TABLE_PL = pl.from_pandas(FIX_EVALUATE_NONTARGET_BINNED_OTU_TABLE_PD)
FIX_EVALUATE_NONTARGET_ELUSIVE_CLUSTERS = FIX_EVALUATE_NONTARGET_ELUSIVE_CLUSTERS_PD
FIX_EVALUATE_NONTARGET_ELUSIVE_OTU_TABLE = FIX_EVALUATE_NONTARGET_ELUSIVE_OTU_TABLE_PD
FIX_EVALUATE_NONTARGET_BINNED_OTU_TABLE = FIX_EVALUATE_NONTARGET_BINNED_OTU_TABLE_PD

# --- evaluate_unbinned ---
FIX_EVALUATE_UNBINNED_PD = pd.DataFrame({"gene": ["g1", "g1", "g2"], "sequence": ["s1", "s1", "s2"], "target": ["t1", None, "t2"], "taxonomy": ["tax1", "tax_missing", "tax2"]})
FIX_EVALUATE_UNBINNED_PL = pl.from_pandas(FIX_EVALUATE_UNBINNED_PD)

print("✅ Fixtures loaded")


In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_evaluate_coassembly_edges(elusive_clusters, elusive_edges=None):
    if elusive_edges is None:
        elusive_edges = pd.DataFrame({"sample1":["s1.1"],"sample2":["s2.1"],"coassembly":["c1"],"target_ids":["t1"]})
    elusive_edges["sample1"] = elusive_edges["sample1"].apply(lambda x: re.sub(r"\.1$", "", x))
    elusive_edges["sample2"] = elusive_edges["sample2"].apply(lambda x: re.sub(r"\.1$", "", x))
    elusive_edges = (elusive_edges
        .set_index("sample1")
        .join(elusive_clusters)
        .reset_index()
        .rename(columns={"index": "sample1"})
        .set_index("sample2")
        .join(elusive_clusters, rsuffix="2")
        .reset_index()
        .rename(columns={"index": "sample2"})
        )
    coassembly_edges = elusive_edges[elusive_edges["coassembly"] == elusive_edges["coassembly2"]].copy()
    coassembly_edges["target"] = coassembly_edges["target_ids"].str.split(",")
    coassembly_edges = coassembly_edges.explode("target").drop_duplicates(["target", "coassembly"]).set_index("target")[["coassembly"]]
    return coassembly_edges

def before_evaluate_combined(haystack_otu_table, recovered_coassemblies, recovered_otu_table, a):
    combined_otu_table = recovered_otu_table.set_index(["coassembly", "gene", "sequence"])[["genome", "taxonomy"]].join(haystack_otu_table, how="outer", rsuffix="old").reset_index()
    combined_otu_table["taxonomy"] = combined_otu_table["taxonomy"].combine(combined_otu_table["taxonomyold"], lambda a,b: a if not pd.isna(a) else b)
    combined_otu_table = combined_otu_table.drop("taxonomyold", axis=1)
    combined_otu_table = combined_otu_table[combined_otu_table["coassembly"].isin(recovered_coassemblies)]
    combined_otu_table = combined_otu_table[combined_otu_table["genome"].notnull() | combined_otu_table["target"].notnull()]
    return combined_otu_table

def before_evaluate_nontarget(elusive_clusters, elusive_otu_table, binned_otu_table):
    binned_otu_table["sample"] = binned_otu_table["sample"].apply(lambda x: re.sub(r"\.1$", "", x))
    nontarget_otu_table = (binned_otu_table
        .set_index(["sample"])[["gene", "sequence", "taxonomy", "found_in"]]
        .join(elusive_clusters)
        .dropna(subset=["coassembly"])
        .drop_duplicates()
        .set_index(["coassembly", "gene", "sequence"])
        )
    nontarget_otu_table["target"] = None

    haystack_otu_table = pd.concat([elusive_otu_table, nontarget_otu_table])
    return haystack_otu_table

def before_evaluate_unbinned(unbinned_otu_table=None):
    if unbinned_otu_table is None:
        unbinned_otu_table = pd.DataFrame({"gene":["g1"],"sequence":["s1"],"target":["t1"],"taxonomy":["tax1"]})
    unbinned_otu_table = unbinned_otu_table.groupby(["gene", "sequence"]).first()[["target", "taxonomy"]].reset_index()
    unbinned_otu_table.dropna(inplace=True)
    unbinned_otu_table["target"] = unbinned_otu_table["target"].astype(str)
    return unbinned_otu_table

In [ ]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_evaluate_coassembly_edges(elusive_clusters, elusive_edges=None):
    if elusive_edges is None:
        elusive_edges = pl.DataFrame({"sample1":["s1.1"],"sample2":["s2.1"],"coassembly":["c1"],"target_ids":["t1"]})
    import re

    elusive_edges = elusive_edges.with_columns([
        pl.col("sample1").cast(pl.Utf8).str.replace(r"\.1$", ""),
        pl.col("sample2").cast(pl.Utf8).str.replace(r"\.1$", ""),
    ])

    _key = elusive_clusters.columns[0]

    elusive_edges = elusive_edges.join(
        elusive_clusters,
        left_on="sample1",
        right_on=_key,
        how="left",
    )

    elusive_edges = elusive_edges.join(
        elusive_clusters,
        left_on="sample2",
        right_on=_key,
        how="left",
        suffix="2",
    )

    coassembly_edges = elusive_edges.filter(
        pl.col("coassembly") == pl.col("coassembly2")
    ).clone()

    coassembly_edges = (
        coassembly_edges.with_columns(
            pl.col("target_ids").cast(pl.Utf8).str.split(",").alias("target")
        )
        .explode("target")
        .unique(subset=["target", "coassembly"], keep="first")
        .select(["target", "coassembly"])
    )
    return coassembly_edges

def gen_evaluate_combined(haystack_otu_table, recovered_coassemblies, recovered_otu_table, a):

    combined_otu_table = (
        recovered_otu_table.select(["coassembly", "gene", "sequence", "genome", "taxonomy"])
        .join(haystack_otu_table, how="outer", on=["coassembly", "gene", "sequence"], suffix="old")
    )
    combined_otu_table = combined_otu_table.with_columns(
        pl.col("taxonomy").fill_null(pl.col("taxonomyold")).alias("taxonomy")
    )
    combined_otu_table = combined_otu_table.drop("taxonomyold")
    combined_otu_table = combined_otu_table.filter(pl.col("coassembly").is_in(recovered_coassemblies))
    combined_otu_table = combined_otu_table.filter(pl.col("genome").is_not_null() | pl.col("target").is_not_null())
    return combined_otu_table

def gen_evaluate_nontarget(elusive_clusters, elusive_otu_table, binned_otu_table):
    import re

    binned_otu_table = binned_otu_table.with_columns(
        pl.col("sample").cast(pl.Utf8).str.replace(r"\.1$", "")
    )

    nontarget_otu_table = (
        binned_otu_table
        .join(elusive_clusters, on="sample", how="inner")
        .drop_nulls(subset=["coassembly"])
        .unique()
    )

    nontarget_otu_table = nontarget_otu_table.with_columns(pl.lit(None).alias("target"))

    haystack_otu_table = pl.concat([elusive_otu_table, nontarget_otu_table], how="diagonal")
    return haystack_otu_table

def gen_evaluate_unbinned(unbinned_otu_table=None):
    if unbinned_otu_table is None:
        unbinned_otu_table = pl.DataFrame({"gene":["g1"],"sequence":["s1"],"target":["t1"],"taxonomy":["tax1"]})

    unbinned_otu_table = (
        unbinned_otu_table.group_by(["gene", "sequence"]).agg(pl.all().first()).select(
            ["gene", "sequence", "target", "taxonomy"]
        )
    )
    unbinned_otu_table = unbinned_otu_table.drop_nulls()
    unbinned_otu_table = unbinned_otu_table.with_columns(pl.col("target").cast(pl.Utf8))
    return unbinned_otu_table

In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: evaluate_combined ===

try:
    _r = gen_evaluate_combined(FIX_EVALUATE_COMBINED_HAYSTACK_OTU_TABLE_PL, FIX_EVALUATE_COMBINED_RECOVERED_COASSEMBLIES, FIX_EVALUATE_COMBINED_RECOVERED_OTU_TABLE_PL, FIX_EVALUATE_COMBINED_A)
    print("✅ L1 smoke gen_evaluate_combined: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_evaluate_combined: {type(_e).__name__}: {_e}")

try:
    _rb = before_evaluate_combined(FIX_EVALUATE_COMBINED_HAYSTACK_OTU_TABLE_PD, FIX_EVALUATE_COMBINED_RECOVERED_COASSEMBLIES, FIX_EVALUATE_COMBINED_RECOVERED_OTU_TABLE_PD, FIX_EVALUATE_COMBINED_A)
    print("✅ L1 smoke before_evaluate_combined: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_evaluate_combined: {type(_e).__name__}: {_e}")

try:
    _rb = before_evaluate_combined(FIX_EVALUATE_COMBINED_HAYSTACK_OTU_TABLE_PD, FIX_EVALUATE_COMBINED_RECOVERED_COASSEMBLIES, FIX_EVALUATE_COMBINED_RECOVERED_OTU_TABLE_PD, FIX_EVALUATE_COMBINED_A)
    _rg = gen_evaluate_combined(FIX_EVALUATE_COMBINED_HAYSTACK_OTU_TABLE_PL, FIX_EVALUATE_COMBINED_RECOVERED_COASSEMBLIES, FIX_EVALUATE_COMBINED_RECOVERED_OTU_TABLE_PL, FIX_EVALUATE_COMBINED_A)
    compare(_rb, _rg, "evaluate_combined")
except Exception as _e:
    print(f"❌ L2 equivalence evaluate_combined: setup error — {type(_e).__name__}: {_e}")

try:
    _rb = before_evaluate_combined(
        FIX_EVALUATE_COMBINED_HAYSTACK_OTU_TABLE_PD.head(0),
        [],
        FIX_EVALUATE_COMBINED_RECOVERED_OTU_TABLE_PD.head(0),
        FIX_EVALUATE_COMBINED_A,
    )
    _rg = gen_evaluate_combined(
        FIX_EVALUATE_COMBINED_HAYSTACK_OTU_TABLE_PL.head(0),
        [],
        FIX_EVALUATE_COMBINED_RECOVERED_OTU_TABLE_PL.head(0),
        FIX_EVALUATE_COMBINED_A,
    )
    compare(_rb, _rg, "L3 edge evaluate_combined empty", check_row_order=True)
except Exception as _e:
    print(f"❌ L3 edge evaluate_combined empty: {type(_e).__name__}: {_e}")
